# Diagnostic linear probe (appearance / skeleton)

Trains ONLY a linear layer `Linear(dim -> classes)` on the frozen features, with CTC,
in the same skeleton vocabulary used by the fusion. It answers: **why does the
fusion's appearance head stall at ~52% instead of ~39%?**

- **app** probe ~39-42%  -> features and labels are fine; it is the FUSION that overfits.
- **app** probe ~52%     -> upstream feature-label misalignment (vocab/keys).
- **skel** probe ~42%    -> sanity check: vocab, labels and decoding all work.

In [ ]:
# --- setup path + cuDNN off ---
import os, sys, torch
torch.backends.cudnn.enabled = False

_here = os.path.abspath('.')
CODE = None
for _k in range(6):
    _b = os.path.abspath(os.path.join(_here, *(['..'] * _k)))
    if os.path.exists(os.path.join(_b, 'csrl_skeleton', 'model.py')):
        CODE = _b; break
assert CODE, 'non trovo code/csrl_skeleton'
DIST = os.path.join(CODE, 'distillation')
FUS = os.path.join(DIST, 'experiments', '02_fusion')
for p in (FUS, os.path.join(CODE, 'csrl_skeleton'), DIST):
    if p not in sys.path:
        sys.path.insert(0, p)
print('FUS:', FUS)

In [ ]:
# --- probe APPEARANCE (il test chiave) ---
from probe_appearance import main as probe
app_wer = probe('app', epochs=30)
print('\n>>> probe APPEARANCE best dev:', round(app_wer, 2), '%')

In [ ]:
# --- probe SKELETON (sanity ~42%) ---
from probe_appearance import main as probe
skel_wer = probe('skel', epochs=30)
print('\n>>> probe SKELETON best dev:', round(skel_wer, 2), '%')

In [ ]:
# --- late fusion leggera + ORACLE di complementarita' (il test che decide) ---
# Allena le due teste lineari (best-on-dev, niente overfit) e riporta su dev+test:
#   app / skel / ORACLE (limite superiore del routing) / w-sum log-lineare / conf-gate.
# ORACLE << 43 -> c'e' complementarita', il gating puo' vincere.
# ORACLE ~ 43  -> stream ridondanti, fusione neutra come la distillazione.
from late_fusion import main as late_fusion
late_fusion(epochs=25)


In [ ]:
# --- routing a livello di SEQUENZA (la tua idea), multi-seed ---
# Oracle ~36% (dev) = tetto del "scegli per-frase quale stream fidarsi".
# conf-route (1 param) e' il vincitore: ~1.9pt sotto il miglior stream singolo.
# 3 seed -> media +/- std, per confermare che il guadagno non e' rumore.
from seq_routing import main as seq_routing
seq_routing(epochs=25, seeds=(42, 123, 2024))
